## Setup

You will need to make a copy of this notebook in your Google Drive before you can edit the homework files. You can do so with **File &rarr; Save a copy in Drive**.
Then run the setup code below to install the requirements to run the lab.

In [ ]:
#@title mount your Google Drive
#@markdown Your work will be stored in a folder called `lab4_release` by default to prevent Colab instance timeouts from deleting your edits.

import os
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
#@title set up mount symlink

DRIVE_PATH = '/content/gdrive/My\ Drive/lab4_release'
DRIVE_PYTHON_PATH = DRIVE_PATH.replace('\\', '')
if not os.path.exists(DRIVE_PYTHON_PATH):
  %mkdir $DRIVE_PATH

## the space in `My Drive` causes some issues,
## make a symlink to avoid this
SYM_PATH = '/content/lab4_release'
if not os.path.exists(SYM_PATH):
  !ln -s $DRIVE_PATH $SYM_PATH

In [ ]:
#@title apt install requirements

#@markdown Run each section with Shift+Enter

#@markdown Double-click on section headers to show code.

!apt update
!apt install -y --no-install-recommends \
        build-essential \
        curl \
        git \
        gnupg2 \
        make \
        cmake \
        ffmpeg \
        swig \
        libz-dev \
        unzip \
        zlib1g-dev \
        libglfw3 \
        libglfw3-dev \
        libxrandr2 \
        libxinerama-dev \
        libxi6 \
        libxcursor-dev \
        libgl1-mesa-dev \
        libgl1-mesa-glx \
        libglew-dev \
        libosmesa6-dev \
        lsb-release \
        ack-grep \
        patchelf \
        wget \
        xpra \
        xserver-xorg-dev \
        xvfb \
        ffmpeg

In [ ]:
#@title install mujoco

!pip install mujoco gymnasium gymnasium-robotics
!pip install protobuf==3.20.3 # Downgrade protobuf to avoid compatibility issues with tensorboardX

In [ ]:
#@title verify mujoco install

import mujoco
print('mujoco version:', mujoco.__version__)

No license key required. The `mujoco` pip package includes the MuJoCo binaries directly.

In [ ]:
#@title upload and install lab4 release

## Upload lab4_release.zip using the Colab file browser, then run this cell
%cd $SYM_PATH
!unzip -o -q /content/gdrive/MyDrive/lab4_release.zip
%cd lab4_release
!sed -i 's/matplotlib==2.2.2/matplotlib/g' requirements_colab.txt
!sed -i 's/torch==1.6.0/torch/g' requirements_colab.txt
!sed -i 's/opencv-python==4.4.0.42/opencv-python/g' requirements_colab.txt
!sed -i '/box2d-py/d' requirements_colab.txt
!sed -i 's/ipython==6.4.0/ipython/g' requirements_colab.txt
!sed -i 's/tensorboard==2.3.0/tensorboard/g' requirements_colab.txt
!sed -i '/^gym==0.17.2/d' requirements_colab.txt # Remove specific gym version to allow gymnasium to be used
%pip install -r requirements_colab.txt
%pip install -e .

Great! We have successfully installed the environments. To test it out, we would like to show you a video output of an example trajectory from random and the expert data provided in the code pipeline.

MuJoCo comes with a variety of continuous control tasks, including:
*   **Ant**
*   **HalfCheetah**
*   **Hopper**
*   **Humanoid**
*   **Walker2d**

For this assignment, we have chosen the **Ant** task for you to implement. Feel free to explore around and train on other tasks after you finish your Lab 4!

In [ ]:
#@title set up virtual display

from pyvirtualdisplay import Display

display = Display(visible=0, size=(1400, 900))
display.start()

In [ ]:
import os
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from lab4.infrastructure.colab_utils import show_video
# Clear previous videos to avoid confusion
os.system('rm -rf ./video/*')

# Use RecordVideo instead of the outdated wrap_env
env = gym.make("Ant-v4", render_mode="rgb_array", use_contact_forces=True)
env = RecordVideo(env, video_folder='./video')

observation, _ = env.reset()

for i in range(100):
    # env.render(mode='rgb_array')
    obs, rew, term, trunc, info = env.step(env.action_space.sample() )
    if term:
      break;

env.close()
print('Loading video of expert behavior...')
show_video()

It is random sampling actions from the actions space. The ant doesn't seem to be making much progess as it does not move forward.

Now let us try to load the expert data that we will be using to learn form and copy their behaviors -> thus the term Behavior Cloning.
This is a supervised learning task where we are given some data that we want to learn form. But what if we don't have reference data to learn from but only have access to the enviornment? These are the questions that the field Reinforcement Learning or Deep Reinforcement Learning try to answer.

Today we will be working only on Supervised tasks




In [ ]:
import os
import pickle
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from lab4.infrastructure.colab_utils import show_video

# Clear previous videos to avoid confusion
os.system('rm -rf ./video/*')
expert_data_path = 'lab4/expert_data/expert_data_Ant-v2.pkl'
with open(expert_data_path, 'rb') as f:
    expert_data = pickle.load(f)

env = gym.make("Ant-v4", render_mode="rgb_array", use_contact_forces=True)
env = RecordVideo(env, video_folder='./video')

observation, _ = env.reset()

# Play back the first expert trajectory if the loaded data has the expected format.
if len(expert_data) > 0 and 'action' in expert_data[0]:
    trajectory_actions = expert_data[0]['action']

    for i, action_step in enumerate(trajectory_actions):
        if i >= 500:
            break

        action = action_step.squeeze()
        observation, reward, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break
else:
    print("Expert data is not in the expected format (list of trajectory dicts with an action key).")

env.close()
print('Loading video of expert behavior...')
show_video()

In the MuJoCo **Ant** task, the goal is to coordinate a four-legged robot to move forward as quickly and stably as possible.

If your video is generated, you can see that the plan that the agent is doing here is it is using 2 legs that are 2 sides of each other to move forward, while keeping one on the ground to adjust the angles and the other one static.

If your Behavior Cloning (BC) training is successful, your trained agent will learn to mimic this exact same strategy!

## Editing Code
To edit code, click the folder icon on the left menu. Navigate to the corresponding file (`lab4_release/...`). Double click a file to open an editor. There is a timeout of about ~12 hours with Colab while it is active (and less if you close your browser window). We sync your edits to Google Drive so that you won't lose your work in the event of an instance timeout, but you will need to re-mount your Google Drive and re-install packages with every new instance.

We recommend that you look at the instruction pdf file before begin to the coding section. Once done implementing, you can start training your BC agent.

In [ ]:
import os
import time
import numpy as np
import sys
import gymnasium
import gymnasium_robotics

from lab4.infrastructure.rl_trainer import RL_Trainer
from lab4.agents.bc_agent import BCAgent


In [ ]:
# Create a compatibility wrapper for old gym API
class GymCompatWrapper(gymnasium.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        # Ensure metadata has 'video.frames_per_second' for legacy compatibility
        fps = self.env.metadata.get('render_fps', 30)
        self.metadata['video.frames_per_second'] = fps

    def seed(self, seed=None):
        # Gymnasium handles seeding in reset, but we provide this for compatibility
        self.reset(seed=seed)
        return [seed]

    def reset(self, **kwargs):
        # Return standard gymnasium format: (obs, info)
        obs, info = super().reset(**kwargs)
        return obs, info

    def step(self, action):
        # Return standard gymnasium format: (obs, reward, terminated, truncated, info)
        obs, reward, terminated, truncated, info = super().step(action)
        return obs, reward, terminated, truncated, info

# Patch gymnasium.make to automatically wrap the environment
if not hasattr(gymnasium, '_original_make'):
    gymnasium._original_make = gymnasium.make

def _compat_make(*args, **kwargs):
    # Fix Ant-v4 observation space mismatch (111 expected vs 27 default)
    if len(args) > 0 and args[0] == 'Ant-v4':
        kwargs['use_contact_forces'] = True

    env = gymnasium._original_make(*args, **kwargs)
    return GymCompatWrapper(env)

gymnasium.make = _compat_make

# Trick the environment into using gymnasium when it asks for gym
sys.modules['gym'] = gymnasium

# Reload the module manually since autoreload is broken in Python 3.12 for this older IPython version
import importlib
modules_to_reload = [
    'lab4.policies.MLP_policy',
    'lab4.agents.bc_agent',
    'lab4.infrastructure.rl_trainer'
]
for mod in modules_to_reload:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from lab4.infrastructure.rl_trainer import RL_Trainer
from lab4.agents.bc_agent import BCAgent

# %load_ext autoreload
# %autoreload 2


In [ ]:
#@title runtime arguments

class Args:

  def __getitem__(self, key):
    return getattr(self, key)

  def __setitem__(self, key, val):
    setattr(self, key, val)

  #@markdown expert data
  expert_data = 'lab4/expert_data/expert_data_Ant-v2.pkl' #@param
  env_name = 'Ant-v4' #@param ['Ant-v4', 'Humanoid-v4', 'Walker2d-v4', 'HalfCheetah-v4', 'Hopper-v4']
  exp_name = 'test_bc_ant' #@param
  ep_len = 1000 #@param {type: "integer"}
  save_params = True #@param {type: "boolean"}

  # Keep n_iter=1 for plain BC. If you want more optimization,
  # increase num_agent_train_steps_per_iter instead of collecting
  # additional on-policy rollouts.
  num_agent_train_steps_per_iter = 100 #@param {type: "integer"})
  n_iter = 1 #@param {type: "integer"})

  #@markdown batches & buffers
  batch_size = 1000 #@param {type: "integer"})
  eval_batch_size = 1000 #@param {type: "integer"}
  train_batch_size = 100 #@param {type: "integer"}
  max_replay_buffer_size = 1000000 #@param {type: "integer"}

  #@markdown network
  n_layers = 1 #@param {type: "integer"}
  size = 32 #@param {type: "integer"}
  learning_rate = 1e-4 #@param {type: "number"}

  #@markdown logging
  video_log_freq = 5 #@param {type: "integer"}
  scalar_log_freq = 1 #@param {type: "integer"}

  #@markdown gpu & run-time settings
  no_gpu = False #@param {type: "boolean"}
  which_gpu = 0 #@param {type: "integer"}
  seed = 1 #@param {type: "integer"}

args = Args()


In [ ]:
#@title define `BC_Trainer`
class BC_Trainer(object):

    def __init__(self, params):
        #######################
        ## AGENT PARAMS
        #######################

        agent_params = {
            'n_layers': params['n_layers'],
            'size': params['size'],
            'learning_rate': params['learning_rate'],
            'max_replay_buffer_size': params['max_replay_buffer_size'],
            }

        self.params = params
        self.params['agent_class'] = BCAgent ## TODO: look in here and implement this
        self.params['agent_params'] = agent_params

        ################
        ## RL TRAINER
        ################

        self.rl_trainer = RL_Trainer(self.params) ## TODO: look in here and implement this

    def run_training_loop(self):

        self.rl_trainer.run_training_loop(
            n_iter=self.params['n_iter'],
            initial_expertdata=self.params['expert_data'],
            collect_policy=self.rl_trainer.agent.actor,
            eval_policy=self.rl_trainer.agent.actor,
        )


In [ ]:
#@title create directory for logging

logdir_prefix = 'q1_'

data_path ='/content/lab4_release/data'
if not (os.path.exists(data_path)):
    os.makedirs(data_path)
logdir = logdir_prefix + args.exp_name + '_' + args.env_name + \
         '_' + time.strftime("%d-%m-%Y_%H-%M-%S")
logdir = os.path.join(data_path, logdir)
args['logdir'] = logdir
if not(os.path.exists(logdir)):
    os.makedirs(logdir)

In [ ]:
## run training
print(args.logdir)

# We can now just run the trainer directly like your PC script
trainer = BC_Trainer(args)
trainer.run_training_loop()

Awesome! Now your training works! To visualize and see if your policy is doing good or not, here is a script that can help you visualize the video output from your recently trained checkpoint! Then to make the policy better, feel free to finetune the params!

## Debug: Checkpoint Summary and Video

The next cell reads the final row of `progress.csv` to print the latest logged training loss and evaluation return.
The video export cell renders an `.mp4` from the latest saved `policy_itr_*.pt` checkpoint in `args.logdir`, so it matches what you saved during training.


In [ ]:
#@title print final logged metrics from progress.csv
import csv
import glob
import os

progress_path = os.path.join(args.logdir, 'progress.csv')
if not os.path.exists(progress_path):
    raise FileNotFoundError(f'Could not find progress.csv at {progress_path}')

with open(progress_path, 'r') as f:
    rows = list(csv.DictReader(f))

if not rows:
    raise ValueError('progress.csv is empty')

final_row = rows[-1]

def maybe_float(key):
    value = final_row.get(key, '')
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

training_loss = maybe_float('Training Loss')
if training_loss is None:
    training_loss = maybe_float('Training_Loss')
eval_return = maybe_float('Eval_AverageReturn')
expert_return = maybe_float('Initial_DataCollection_AverageReturn')

checkpoint_files = sorted(glob.glob(os.path.join(args.logdir, 'policy_itr_*.pt')))
latest_checkpoint = checkpoint_files[-1] if checkpoint_files else None

print(f'Run directory: {args.logdir}')
if latest_checkpoint:
    print(f'Latest checkpoint: {latest_checkpoint}')
else:
    print('No saved policy checkpoint found. Make sure save_params=True.')

if training_loss is not None:
    print(f'Final Training Loss: {training_loss:.6f}')
else:
    print('Training Loss was not found in progress.csv.')

if eval_return is not None:
    print(f'Final Eval Return: {eval_return:.2f}')
else:
    print('Eval_AverageReturn was not found in progress.csv.')

if expert_return is not None:
    print(f'Expert Return: {expert_return:.2f}')

if expert_return is not None and eval_return is not None and expert_return != 0:
    print(f'Percent of Expert: {100.0 * eval_return / expert_return:.1f}%')


In [ ]:
#@title export and display a video from the saved checkpoint
import base64
import glob
import io
import os
import subprocess
import sys
from IPython.display import HTML, display

video_output_dir = os.path.join(args.logdir, 'exported_videos')
os.makedirs(video_output_dir, exist_ok=True)

subprocess.run([
    sys.executable,
    'lab4/scripts/visualize.py',
    '--logdir', args.logdir,
    '--export-video',
    '--no-show',
    '--env-name', args.env_name,
    '--n-layers', str(args.n_layers),
    '--size', str(args.size),
    '--seed', str(args.seed),
    '--video-output-dir', video_output_dir,
], check=True)

mp4_files = sorted(
    glob.glob(os.path.join(video_output_dir, '*.mp4')),
    key=os.path.getmtime,
    reverse=True,
)

if not mp4_files:
    raise FileNotFoundError(f'No exported video found in {video_output_dir}')

latest_video = mp4_files[0]
print(f'Displaying: {latest_video}')

video = io.open(latest_video, 'r+b').read()
encoded = base64.b64encode(video).decode('ascii')
display(HTML(f'''<video alt="trained policy" controls style="height: 400px;">
  <source src="data:video/mp4;base64,{encoded}" type="video/mp4" />
</video>'''))


Congratulations, you now complete a robot learning lab !